# NeuroTrain Lab — Notebook 2: Pérdida y Backpropagation

**Tema:** qué es una función de pérdida, cómo se mide el error con MSE
(regresión) y Cross-Entropy (clasificación), y cómo la regla de la cadena
permite calcular exactamente cuánto debe cambiar cada weight para reducir
ese error — lo que las librerías llaman *backpropagation*.

> Segundo notebook de 4. En el Notebook 1 aprendiste a **predecir** (forward
> propagation). Aquí aprendes a **medir el error** y a calcular la dirección en
> la que hay que mover cada weight para reducirlo. Moverlos de verdad —
> entrenar— es el Notebook 3.

## 🎯 Qué aprenderás en este notebook

Al terminar podrás explicar, sin fórmulas de memoria:

1. Qué es una función de pérdida y por qué no es lo mismo que la exactitud (accuracy).
2. Cómo funciona MSE (Mean Squared Error) para problemas de regresión.
3. Cómo funciona Binary Cross-Entropy para problemas de clasificación, y por qué
   "equivocarse con seguridad" se penaliza mucho más que "no estar seguro".
4. Qué es la regla de la cadena y cómo se aplica, paso a paso, sobre un grafo
   computacional pequeño para obtener un gradiente.
5. Que `loss.backward()` en PyTorch no es magia: es exactamente la misma
   aritmética que acabas de hacer a mano.

**Mapa mental:** `predicción → pérdida → regla de la cadena → gradiente → (Notebook 3: ajustar weights)`

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "breast_cancer_wisconsin.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from neurotrain.celebrations import celebrate

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

print("NumPy:", np.__version__, "| PyTorch:", torch.__version__, "| TensorFlow:", tf.__version__)
print("Raíz del proyecto:", PROJECT_ROOT)

## 1. Qué es una función de pérdida

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 CONCEPTO CLAVE — El juego de 'frío / caliente'</b><br><br>
De niños jugábamos a esconder un objeto y guiar a alguien con "frío" (lejos) o
"caliente" (cerca). Una función de pérdida hace justo eso: convierte "qué tan
equivocada está la predicción" en **un solo número**. Cuanto más alto, más
"frío" está el modelo; cuanto más bajo (idealmente 0), más "caliente".

No nos dice **en qué dirección** moverse — eso lo resuelve la regla de la
cadena, en la Sección 4 — pero sí nos dice si vamos mejorando.
</div>

### Pérdida vs. exactitud (accuracy)

No son lo mismo, y confundirlas es un error típico:

- La **pérdida** (loss) es lo que el optimizador **minimiza directamente**. Es
  continua y sensible: distingue entre "acerté con 0.51 de probabilidad" y
  "acerté con 0.99 de probabilidad", aunque ambas cuenten como acierto.
- La **exactitud** (accuracy) es lo que a los humanos nos resulta fácil de
  interpretar ("acertó el 90% de las veces"), pero es una cuenta más tosca —
  solo mira si cruzaste el umbral de 0.5, no por cuánto.

Por eso pueden **divergir a corto plazo**: la pérdida puede bajar (el modelo
está cada vez más seguro) sin que la exactitud cambie todavía, o viceversa.

<div style="border-left:4px solid #2563EB; background:#EFF6FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>❓ DUDA PROBABLE — Entonces, ¿por qué no entrenar directamente para maximizar la exactitud?</b><br><br>
Porque la exactitud no es **derivable**: es un escalón (acierto/fallo), no una
curva suave. La regla de la cadena (Sección 4) necesita una función suave para
calcular en qué dirección mover cada weight. Por eso entrenamos minimizando una
pérdida continua (MSE, Cross-Entropy...) y usamos la exactitud solo para
**interpretar** el resultado, no para optimizarlo.
</div>

## 2. MSE (Mean Squared Error): la pérdida para regresión

Para un problema de **regresión** (predecir un número continuo, no una clase),
la pérdida más común es el **Error Cuadrático Medio**:

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

Elevar al cuadrado hace dos cosas: (1) los errores negativos y positivos no se
cancelan entre sí, y (2) penaliza mucho más los errores grandes que los pequeños.

In [ ]:
# Juguete: predecir la temperatura (°C) real a partir de un termómetro barato
y_real = np.array([20.0, 25.0, 30.0, 22.0])
y_pred = np.array([18.0, 27.0, 29.0, 25.0])

errores_cuadrados = (y_real - y_pred) ** 2
mse = errores_cuadrados.mean()

print("Errores al cuadrado:", errores_cuadrados)
print("MSE:", mse)

plt.figure(figsize=(5.5, 4))
x_pos = np.arange(len(y_real))
plt.scatter(x_pos, y_real, color="#2563EB", label="Real", zorder=3, s=60)
plt.scatter(x_pos, y_pred, color="#F97316", label="Predicción", zorder=3, s=60)
for xi, real, pred in zip(x_pos, y_real, y_pred):
    plt.plot([xi, xi], [real, pred], color="#94A3B8", linestyle="--", zorder=1)
plt.title(f"Brecha al cuadrado entre predicción y realidad (MSE = {mse:.2f})")
plt.xticks(x_pos, [f"medición {i+1}" for i in x_pos])
plt.legend()
plt.grid(alpha=0.2)
plt.show()

### ✏️ Ejercicio

Implementa `mse(y_true, y_pred)` manualmente (sin usar `tf.keras.losses`) y
comprueba que coincide con `tf.keras.losses.MeanSquaredError()` sobre los mismos
`y_real`/`y_pred` de arriba (debería dar `4.5`).

In [ ]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ✏️✏️✏️)


mi_mse = mse(y_real, y_pred)
mse_keras = tf.keras.losses.MeanSquaredError()(y_real, y_pred).numpy()

print("Manual:", mi_mse, "| Keras:", mse_keras)
assert np.isclose(mi_mse, mse_keras)
print("¡Coinciden!")

<details>
<summary><b>Ver solución</b></summary>

```python
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


mi_mse = mse(y_real, y_pred)
mse_keras = tf.keras.losses.MeanSquaredError()(y_real, y_pred).numpy()

print("Manual:", mi_mse, "| Keras:", mse_keras)
assert np.isclose(mi_mse, mse_keras)
print("¡Coinciden!")
```

</details>

## 3. Cross-Entropy: la pérdida para clasificación

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 CONCEPTO CLAVE — MSE no es el ajuste natural para nuestro problema</b><br><br>
Nuestro hilo conductor —¿tumor benigno o maligno?— es una **clasificación
binaria**: la salida es una probabilidad entre 0 y 1, no un número continuo sin
límites. MSE trataría por igual un error de "predije 0.5 cuando era 1" que uno
de "predije 0.99 cuando era 0" en términos relativos — no captura bien que
**estar seguro y equivocado es mucho peor que estar inseguro**. Para eso existe
la **Binary Cross-Entropy (BCE)**.
</div>

Para una etiqueta verdadera $y \in \{0, 1\}$ y una probabilidad predicha $p$:

$$\text{BCE} = -\big[y \log(p) + (1-y)\log(1-p)\big]$$

Si la etiqueta real es $y=1$, esto se reduce a $-\log(p)$: cuanto más lejos esté
$p$ de 1, más grande (y más rápido crece) la pérdida.

In [ ]:
# Tabla numérica: y_real = 1 (maligno), tres niveles de confianza del modelo
bce_keras = tf.keras.losses.BinaryCrossentropy()
y_real_bce = tf.constant([[1.0]])

probabilidades = [0.9, 0.5, 0.1]
perdidas = [bce_keras(y_real_bce, tf.constant([[p]])).numpy() for p in probabilidades]

for p, l in zip(probabilidades, perdidas):
    print(f"p={p:>3} (confianza {'correcta' if p > 0.5 else 'incorrecta' if p < 0.5 else 'nula'})"
          f"  ->  BCE = {l:.4f}")

plt.figure(figsize=(5, 3.8))
plt.bar([str(p) for p in probabilidades], perdidas, color=["#22C55E", "#F97316", "#F43F5E"])
plt.title("BCE para y=1 según la probabilidad predicha")
plt.xlabel("p predicho")
plt.ylabel("Binary Cross-Entropy")
plt.grid(alpha=0.2, axis="y")
plt.show()

Con `y=1`: `p=0.9` (correcto y seguro) da BCE ≈ **0.105**; `p=0.5` (inseguro) da
BCE ≈ **0.693**; `p=0.1` (seguro y **equivocado**) da BCE ≈ **2.303** — más de 20
veces la pérdida de `p=0.9`. Esa asimetría es intencional: el modelo aprende a
**no estar confiadamente equivocado**.

### ✏️ Ejercicio

Implementa `binary_cross_entropy(y_true, y_pred)` manualmente usando
`-[y·log(p) + (1-y)·log(1-p)]`, y comprueba que coincide con
`tf.keras.losses.BinaryCrossentropy()` para `y=1, p=0.9`.

In [ ]:
def binary_cross_entropy(y_true, y_pred):
    return -(y_true * np.log(y_pred) + (1 - y_true) * np.log(✏️✏️✏️))


mi_bce = binary_cross_entropy(np.array([1.0]), np.array([0.9]))
bce_keras_val = bce_keras(tf.constant([[1.0]]), tf.constant([[0.9]])).numpy()

print("Manual:", mi_bce, "| Keras:", bce_keras_val)
assert np.isclose(mi_bce, bce_keras_val, atol=1e-5)
print("¡Coinciden!")

<details>
<summary><b>Ver solución</b></summary>

```python
def binary_cross_entropy(y_true, y_pred):
    return -(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


mi_bce = binary_cross_entropy(np.array([1.0]), np.array([0.9]))
bce_keras_val = bce_keras(tf.constant([[1.0]]), tf.constant([[0.9]])).numpy()

print("Manual:", mi_bce, "| Keras:", bce_keras_val)
assert np.isclose(mi_bce, bce_keras_val, atol=1e-5)
print("¡Coinciden!")
```

</details>

<div style="border-left:4px solid #F97316; background:#FFF7ED; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>⚠️ ERROR TÍPICO — np.log(0) rompe todo</b><br><br>
Si `p` llega a ser exactamente 0 o 1, `np.log(0)` da `-inf` y la pérdida explota.
Keras evita esto internamente recortando `p` a un rango seguro (ej.
`[1e-7, 1-1e-7]`). No lo necesitamos en estos ejemplos porque elegimos `p`
estrictamente entre 0 y 1, pero es la razón por la que nunca deberías usar
Sigmoid + BCE manual sin ese recorte en código de producción.
</div>

<div style="text-align:center; opacity:.85; font-style:italic; margin:1.1rem 0; font-size:1.05rem;">🔥 Mitad del camino: de aquí a nada desentrañarás los misterios de la mente artificial.</div>

## 4. La regla de la cadena: cómo se calcula un gradiente

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 CONCEPTO CLAVE — El corazón de este notebook</b><br><br>
Ya sabemos **medir** el error (Secciones 2-3). Ahora necesitamos saber **en qué
dirección y cuánto** mover cada weight para reducirlo. Eso es un **gradiente**:
la derivada de la pérdida respecto a un weight, `dL/dw`. La regla de la cadena
nos deja calcularlo paso a paso, multiplicando derivadas locales a lo largo del
camino desde la pérdida hasta el weight — eso es *backpropagation*.
</div>

### Ejemplo mínimo: una sola entrada, un solo weight

$$x = 2.0 \qquad w = 0.5 \qquad \text{pred} = x \cdot w \qquad \text{target} = 2.0
\qquad L = (\text{pred} - \text{target})^2$$

Grafo computacional (hacia adelante):

```
w --(× x)--> pred --(vs target)--> L
```

Y hacia atrás (lo que queremos: `dL/dw`):

```
L --> pred --> w
```

Paso a paso, con regla de la cadena `dL/dw = (dL/dpred) · (dpred/dw)`:

1. `pred = x · w = 2.0 · 0.5 = 1.0`
2. `L = (pred - target)² = (1.0 - 2.0)² = 1.0`
3. `dL/dpred = 2·(pred - target) = 2·(-1.0) = -2.0`
4. `dpred/dw = x = 2.0`
5. `dL/dw = dL/dpred · dpred/dw = -2.0 · 2.0 = -4.0`

`dL/dw = -4.0` significa: si aumentamos `w` un poquito, la pérdida **baja**
(gradiente negativo) — así que el optimizador (Notebook 3) moverá `w` en
dirección **contraria** al gradiente para reducir `L`.

In [ ]:
# Verificamos el ejemplo anterior con código, sin autograd todavía — puro cálculo manual
x, w, target = 2.0, 0.5, 2.0

pred = x * w
loss = (pred - target) ** 2

dL_dpred = 2 * (pred - target)
dpred_dw = x
dL_dw = dL_dpred * dpred_dw

print(f"pred={pred}  loss={loss}")
print(f"dL/dpred={dL_dpred}  dpred/dw={dpred_dw}  dL/dw={dL_dw}")

### Un grafo un poco más grande: entrada → lineal → activación → pérdida

Añadimos un bias y una activación ReLU antes de calcular la pérdida —
exactamente la estructura de una neurona real seguida de su pérdida:

$$x=3.0 \quad w=0.4 \quad b=-0.5 \quad z = x\cdot w + b \quad \text{pred} = \text{ReLU}(z)
\quad \text{target}=1.5 \quad L=(\text{pred}-\text{target})^2$$

Forward, paso a paso:

1. `z = x·w + b = 3.0·0.4 + (-0.5) = 0.7`
2. `pred = ReLU(z) = ReLU(0.7) = 0.7` (z es positivo, ReLU no cambia nada)
3. `L = (pred - target)² = (0.7 - 1.5)² = 0.64`

Backward, paso a paso (regla de la cadena en cada nodo, de atrás hacia adelante):

4. `dL/dpred = 2·(pred - target) = 2·(-0.8) = -1.6`
5. `dpred/dz = 1` si `z > 0`, `0` si `z < 0` (la derivada de ReLU) → aquí `z=0.7>0`, así que `dpred/dz = 1`
6. `dL/dz = dL/dpred · dpred/dz = -1.6 · 1 = -1.6`
7. `dz/dw = x = 3.0` → `dL/dw = dL/dz · dz/dw = -1.6 · 3.0 = -4.8`
8. `dz/db = 1` → `dL/db = dL/dz · dz/db = -1.6`

In [ ]:
# Mismo grafo (lineal + ReLU + pérdida), verificado en código
def relu(z):
    return max(0.0, z)


x2, w2, b2, target2 = 3.0, 0.4, -0.5, 1.5

z2 = x2 * w2 + b2
pred2 = relu(z2)
loss2 = (pred2 - target2) ** 2

dL_dpred2 = 2 * (pred2 - target2)
dpred_dz2 = 1.0 if z2 > 0 else 0.0
dL_dz2 = dL_dpred2 * dpred_dz2
dL_dw2 = dL_dz2 * x2
dL_db2 = dL_dz2 * 1.0

print(f"z={z2}  pred={pred2}  loss={loss2}")
print(f"dL/dpred={dL_dpred2}  dpred/dz={dpred_dz2}  dL/dz={dL_dz2}")
print(f"dL/dw={dL_dw2}  dL/db={dL_db2}")

### ✏️ Ejercicio

Repite el mismo grafo (`z = x·w + b`, `pred = ReLU(z)`, `L = (pred - target)²`)
con nuevos números: `x=1.5, w=0.3, b=0.2, target=0.5`. Completa el forward pass
(`z3` y `pred3`) — el resto (backward) ya está escrito para que verifiques tu
resultado.

In [ ]:
x3, w3, b3, target3 = 1.5, 0.3, 0.2, 0.5

z3 = ✏️✏️✏️
pred3 = relu(z3)
loss3 = (pred3 - target3) ** 2

dL_dpred3 = 2 * (pred3 - target3)
dpred_dz3 = 1.0 if z3 > 0 else 0.0
dL_dz3 = dL_dpred3 * dpred_dz3
dL_dw3 = dL_dz3 * x3

print(f"z={z3}  pred={pred3}  loss={loss3}  dL/dw={dL_dw3}")
assert np.isclose(z3, 0.65) and np.isclose(loss3, 0.0225) and np.isclose(dL_dw3, 0.45)
print("¡Correcto!")

<details>
<summary><b>Ver solución</b></summary>

```python
x3, w3, b3, target3 = 1.5, 0.3, 0.2, 0.5

z3 = x3 * w3 + b3
pred3 = relu(z3)
loss3 = (pred3 - target3) ** 2

dL_dpred3 = 2 * (pred3 - target3)
dpred_dz3 = 1.0 if z3 > 0 else 0.0
dL_dz3 = dL_dpred3 * dpred_dz3
dL_dw3 = dL_dz3 * x3

print(f"z={z3}  pred={pred3}  loss={loss3}  dL/dw={dL_dw3}")
assert np.isclose(z3, 0.65) and np.isclose(loss3, 0.0225) and np.isclose(dL_dw3, 0.45)
print("¡Correcto!")
```

</details>

## 5. Autograd: la misma cuenta, calculada por PyTorch

<div style="border-left:4px solid #22C55E; background:#F0FDF4; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>📌 PARA RECORDAR — El momento 'ajá'</b><br><br>
`loss.backward()` **no es magia**: PyTorch recuerda cada operación aplicada a
un tensor con `requires_grad=True` y aplica exactamente la regla de la cadena
que hicimos a mano en la Sección 4. Vamos a reconstruir el ejemplo mínimo
(`x=2.0, w=0.5, target=2.0`) con `torch` y comprobar que `w.grad` da **el mismo
-4.0** que calculamos a mano.
</div>

In [ ]:
xw = torch.tensor(2.0)
w_t = torch.tensor(0.5, requires_grad=True)  # solo w es "entrenable"
target_t = torch.tensor(2.0)

pred_t = xw * w_t
loss_t = (pred_t - target_t) ** 2

loss_t.backward()  # aplica la regla de la cadena automáticamente

print("pred:", pred_t.item(), "| loss:", loss_t.item())
print("w.grad (calculado por autograd):", w_t.grad.item())
print("dL/dw calculado a mano en la Sección 4:", -4.0)
assert np.isclose(w_t.grad.item(), -4.0)
print("\n¡Coinciden exactamente! autograd = regla de la cadena aplicada por software.")

<div style="text-align:center; opacity:.85; font-style:italic; margin:1.1rem 0; font-size:1.05rem;">🚀 La red está tomando forma bajo tus manos.</div>

## 6. Aplicándolo al dataset real: una pérdida, un ejemplo

Cerramos con el hilo conductor del curso: tomamos **una fila real** del dataset
*Breast Cancer Wisconsin* (30 variables), un vector de weights **fijo** (no
entrenado — eso es el Notebook 3) y un bias que elegimos nosotros, calculamos la
probabilidad predicha con Sigmoid y comparamos la Binary Cross-Entropy de
nuestra implementación manual contra `tf.keras.losses.BinaryCrossentropy()`.

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "breast_cancer_wisconsin.csv"
df = pd.read_csv(DATA_PATH)

feature_cols = [c for c in df.columns if c not in ("id", "diagnosis")]
x_row = df.loc[0, feature_cols].to_numpy(dtype="float64")
# diagnosis: 'M' (maligno) -> 1, 'B' (benigno) -> 0
y_row = 1.0 if df.loc[0, "diagnosis"] == "M" else 0.0

# Weights fijos y arbitrarios (sin entrenar): normalizamos x para que z no explote
x_row_norm = (x_row - x_row.mean()) / x_row.std()
rng = np.random.default_rng(RANDOM_STATE)
w_fijo = rng.normal(0, 0.05, size=x_row_norm.shape)
b_fijo = 0.0

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z_row = np.dot(x_row_norm, w_fijo) + b_fijo
p_row = sigmoid(z_row)

bce_manual = binary_cross_entropy(np.array([y_row]), np.array([p_row]))[0]
bce_tf = bce_keras(tf.constant([[y_row]]), tf.constant([[p_row]])).numpy()

print("Etiqueta real (1=maligno, 0=benigno):", y_row)
print("Probabilidad predicha (weights sin entrenar):", p_row)
print("BCE manual:", bce_manual, "| BCE Keras:", bce_tf)
assert np.allclose(bce_manual, bce_tf, atol=1e-5)
print("\n¡Coinciden! Con weights sin entrenar, esta pérdida es solo el punto de partida.")

<div style="border-left:4px solid #2563EB; background:#EFF6FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>❓ DUDA PROBABLE — ¿Por qué la pérdida no es baja si el modelo 'nunca vio' este dato?</b><br><br>
Porque los weights son **aleatorios**, no entrenados — la predicción es
esencialmente una apuesta a ciegas. El punto de esta sección no es obtener una
pérdida baja, sino confirmar que sabemos **calcularla correctamente** para un
caso real. Reducir esa pérdida de verdad, moviendo los weights con el gradiente
que aprendiste a calcular en la Sección 4, es exactamente lo que hace el
optimizador del Notebook 3.
</div>

## 🎯 Autoevaluación

Respóndelas sin mirar atrás. No necesitas frases perfectas: explica el mecanismo con tus palabras.

**1. ¿Cuál es la pérdida más adecuada para un problema de clasificación binaria (¿maligno o benigno?)?**

A. MSE, porque siempre funciona igual de bien
B. Binary Cross-Entropy, porque refleja qué tan correcta es una probabilidad
C. No hace falta ninguna pérdida si ya usamos Sigmoid
D. Accuracy, porque es la métrica que de verdad importa

<details>
<summary><b>Ver respuesta</b></summary>

**B.** MSE trata los errores como distancias numéricas; BCE está diseñada para probabilidades y penaliza con fuerza estar confiadamente equivocado.

</details>

**2. Con y=1, ¿qué predicción produce mayor Binary Cross-Entropy?**

A. p = 0.9
B. p = 0.5
C. p = 0.1
D. Todas producen la misma pérdida

<details>
<summary><b>Ver respuesta</b></summary>

**C.** Cuanto más lejos está p de la etiqueta real (y=1), mayor es -log(p). p=0.1 (confiadamente equivocado) da la pérdida más alta, ≈2.303.

</details>

**3. ¿Qué calcula exactamente `loss.backward()` en PyTorch?**

A. Entrena el modelo y actualiza los weights directamente
B. Aplica la regla de la cadena para calcular el gradiente de la pérdida respecto a cada tensor con requires_grad=True
C. Calcula únicamente la pérdida, sin gradientes
D. Reinicia los weights a valores aleatorios

<details>
<summary><b>Ver respuesta</b></summary>

**B.** backward() recorre el grafo computacional hacia atrás aplicando la regla de la cadena en cada nodo, guardando el resultado en .grad — no mueve los weights por sí solo (eso es el optimizador, Notebook 3).

</details>

**4. Si `dL/dw` da exactamente 0 para un weight, ¿qué implica eso para el entrenamiento en ese punto?**

A. Que el modelo ya es perfecto siempre
B. Que, en ese punto exacto, mover ese weight un poquito no cambiaría la pérdida (puede ser un mínimo, un máximo o un punto de silla)
C. Que hay un error de código y hay que revisar los datos
D. Que ese weight debe eliminarse de la red

<details>
<summary><b>Ver respuesta</b></summary>

**B.** Un gradiente cero solo dice que la pendiente local es plana para ese weight en ese punto — no garantiza que sea el mejor punto posible ni que el resto de la red también esté optimizada.

</details>

**5. En el grafo `w -> (×x) -> pred -> loss`, ¿qué regla te permite obtener `dL/dw` a partir de `dL/dpred` y `dpred/dw`?**

A. La regla de L'Hôpital
B. La regla de la cadena: dL/dw = dL/dpred · dpred/dw
C. La regla de tres simple
D. No se puede calcular sin conocer w

<details>
<summary><b>Ver respuesta</b></summary>

**B.** La regla de la cadena multiplica las derivadas locales a lo largo del camino desde la pérdida hasta el weight — es la base matemática de backpropagation.

</details>

**6. ¿El hecho de que la pérdida baje en un ejemplo concreto garantiza que el modelo, en general, sea mejor?**

<details>
<summary><b>Qué debería incluir una buena respuesta</b></summary>

- Distingue entre la pérdida de un ejemplo individual y la pérdida promedio (o de validación) sobre muchos ejemplos.
- Menciona que mejorar en un ejemplo puede empeorar en otros (sobreajuste a ese punto).
- Concluye que solo la tendencia de la pérdida sobre un conjunto amplio, no un único ejemplo, es una señal fiable.

</details>

**7. Explícale a un compañero la regla de la cadena usando el ejemplo x=2, w=0.5, sin usar fórmulas.**

<details>
<summary><b>Qué debería incluir una buena respuesta</b></summary>

- Describe el camino: cambiar w cambia pred, y cambiar pred cambia la pérdida — el efecto se 'encadena'.
- Explica que el gradiente final es el producto de cuánto cambia cada paso por el cambio del paso anterior.
- Usa el ejemplo concreto (dL/dw = -4.0) para mostrar que el resultado es un número calculable, no una intuición vaga.

</details>

In [ ]:
celebrate(
    "🎉 ¡Enhorabuena! Completaste el Notebook 2: Pérdida y Backpropagation 🎉",
    "Ya sabes medir el error con MSE y Cross-Entropy, y calcular exactamente cómo "
    "moverías cada weight con la regla de la cadena. En el Notebook 3 usarás ese "
    "gradiente de verdad: el optimizador entrena la red paso a paso.",
)